[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/paymantohidifar/deep-learning-for-biology-book/blob/main/notebooks/chapter2-proteins.ipynb)

## **Google Colab Environment Setup**

To run this notebook on Google Colab, follow the following three steps. Otherwise, skip to [Local Environment Setup](#local-environment-setup) to run it locally.

1. **Clone the `dlfb` library**  

   First, clone the repository that contains the `dlfb` library.

In [ ]:
%cd /content
!rm -rf ./dlfb-clone/
!git clone "https://github.com/deep-learning-for-biology/dlfb.git" dlfb-clone --branch main
%cd dlfb-clone

2. **Install dependencies**  
   
   Once the library is cloned, install the required dependencies. Make sure to restart the runtime session because it clears the Python cache so the new versions are actually used.

In [ ]:
%%bash
# Download and execute the 'uv' installer script.
# Add the 'uv' binary directory to your system PATH so you can call 'uv' directly.
# Generate a resolved dependency list (compilation).
# --constraint: Forces versions to align with a specific 'constraints.txt' file.
# Install the compiled dependencies.
# --system: Installs packages into the global system Python rather than a virtual env.

curl -LsSf https://astral.sh/uv/install.sh | sh && \
export PATH="/root/.local/bin:${PATH}" && \
uv pip compile ./requirements/{base,dlfb,proteins,gpu}.txt \
  --color never \
  --constraint ./requirements/constraints.txt | \
uv pip install -r - --system

3. **Provision the datasets**

   You’ll then need to access and download the necessary datasets for this chapter.

In [ ]:
!dlfb-provision --chapter proteins

---

## **Local Environment Setup**

Skip to [Load the `dlfb` Package](#load-the-dlfb-package) if using Google Colab as your runtime.

1. **Verify local environment**

In [ ]:
import sys
from pathlib import Path

active_executable_idx = Path(sys.executable).parts.index("envs")
active_environ = Path(sys.executable).parts[active_executable_idx+1]

assert active_environ == "proteins", "Wrong environment is selected. Update environment to 'proteins'"
print("Correct 'proteins' environment is loaded!")

2. **Provision the datasets locally**

In [ ]:
!dlfb-provision --chapter proteins --destination ../data

---

## **Load the `dlfb` package**  

Finally, load the `dlfb` package.  

> **Note:** Loading can sometimes be finicky on Google Colab. If you encounter issues, simply restart the runtime. All previously downloaded data and installed packages will persist, so you can re-run the load step without repeating everything.

In [ ]:
# Toggle JAX_DISABLE_JIT to True for easier debugging
%env JAX_DISABLE_JIT=True

try:
  import dlfb
except ImportError as exc:
  # NOTE: Packages installed in editable mode are not immediately
  #       recognized by Colab (https://stackoverflow.com/a/63312333).
  import site
  site.main()
  import dlfb

from dlfb.utils.display import display

---

# **Chapter 2. Learning the Language of Proteins**

## **Biology Primer**

This section provides a biological foundation for protein science, emphasizing the deterministic relationship where Sequence $\rightarrow$ Structure $\rightarrow$ Function.

### **Protein Structure**

Protein structure is organized into a four-level hierarchy, determined by the primary sequence of amino acids.

* **Hierarchical Levels**:
    * **Primary**: The linear chain of amino acids.
    * **Secondary**: Localized folding patterns like -helices and -sheets.
    * **Tertiary**: The complete 3D spatial arrangement of a single polypeptide chain.
    * **Quaternary**: The complex assembly of multiple protein subunits (e.g., Hemoglobin).
* **Amino Acid Properties**: There are 20 main amino acids, categorized by biochemical traits such as hydrophobicity, charge (positive/negative), and polarity.
* **The Impact of Mutations**: Even a single amino acid substitution (point mutation) can be catastrophic.
* *Example*: Sickle cell anemia occurs when a hydrophilic amino acid (E) is replaced by a hydrophobic one (V) in hemoglobin.

<div align='center'>
<img src="https://github.com/paymantohidifar/deep-learning-for-biology-book/blob/main/notebooks/images/aa.png" width='750px'>
</div>

### **Protein Function**

Proteins are the primary "workhorses" of the cell. Their functions are systematically categorized using the **Gene Ontology (GO)** framework.
* **Molecular Function**: The specific biochemical activity at the molecular level (e.g., DNA binding or enzyme catalysis).
* **Biological Process**: The larger goal the protein contributes to (e.g., cell division or immune signaling).
* **Cellular Component**: Where the protein is physically located (e.g., mitochondria or nucleus), which often hints at its role.
* **Multi-functionality**: A single protein can have multiple annotations across these categories; for instance, a protein might be a kinase (molecular) that drives muscle contraction (process) in muscle fibers (component).

### **Predicting Protein Function**

Predicting function from sequence is a "grand challenge" because it requires a model to implicitly understand how sequences fold into 3D shapes.
* **Biotechnology & Engineering**: Designing synthetic enzymes for industry or therapeutic proteins for medicine.
* **Disease Analysis**: Identifying how specific mutations (variants) disrupt healthy functions to find therapeutic targets.
* **Genome & Metagenomics**: Assigning functional hypotheses to the millions of "unknown" proteins discovered in new species or environmental samples (like gut bacteria).
* **Computational Strategy**: While full folding prediction is complex, effective workflows often involve using **pretrained embeddings** to capture biological "language" and training lightweight classifiers on top of them.

---

## **Machine Learning Primer**

This section bridges the gap between biological sequences and modern AI, explaining how the "language" of proteins can be parsed using the same techniques that power modern LLMs.

### **Large Language Models**

LLMs are founded on a simple objective that leads to emergent, complex intelligence.

* **Next-Token Prediction**: Models are trained to predict the next character or word (token) based on the preceding context. Variants include **Masked Language Models**, which predict hidden tokens.
* **Emergent Capabilities**: By scaling the number of parameters and the volume of data, models "unsupervisedly" learn to summarize, translate, and reason.
* **Biology as Language**: Because DNA and proteins are sequences from a discrete alphabet with a complex "grammar," they are perfectly suited for LLM architectures.
* **Biological LLMs**: Models like **ESM2** learn rich representations of biological information by being trained on massive corpora of protein sequences.
    
    >**ESM-2:** a state-of-the-art transformer-based protein language model developed by Meta AI (FAIR) that predicts protein structure and function directly from amino acid sequences. Utilizing a BERT-style architecture, it is trained on millions of protein sequences to understand evolutionary, structural, and functional patterns. It enables tasks like mutation effect prediction, protein engineering, and, via ESMFold, rapid 3D structure prediction.


### **Embeddings**

Embeddings are the bridge between raw biological strings and numerical computation.

* **Numerical Vectors**: An embedding is a compact list of floating-point numbers that encodes the "meaning" of a protein.
* **Latent/Semantic Space**: Similar entities (like related proteins) are positioned near each other in an abstract multi-dimensional space. This allows models to identify functional similarities that aren't obvious from the raw sequence alone.
* **Cosine Similarity**: This is the standard metric used to compare how closely aligned two embedding vectors are. It helps rank the most similar known proteins to a new, uncharacterized query.
    
$$\text{cosine similarity}(A,B) = \frac{A.B}{\lVert A \lVert \lVert B \lVert}$$

### **Pretraining and Fine-tuning**

This two-stage process allows models to apply general knowledge to specific, high-stakes tasks.

* **Pretraining**: The model gains "broad knowledge" by training on massive, diverse datasets (e.g., all known protein sequences).
* **Fine-tuning**: A secondary step where the pretrained model is updated on a smaller, specialized dataset to perform a specific task (e.g., predicting toxicity).
* **Frozen Feature Extraction**: An efficient alternative to fine-tuning where the large model remains "frozen" (unchanged). It acts as a feature extractor, providing embeddings that are then fed into a much smaller, custom-trained classifier.

---

## **Representations of Proteins and Protein LMs**

In [ ]:
import py3Dmol
import requests


def fetch_protein_structure(pdb_id: str) -> str:
  """Grab a PDB protein structure from the RCSB Protein Data Bank."""
  url = f"https://files.rcsb.org/download/{pdb_id}.pdb"
  response = requests.get(url)
  return response.text


# The Protein Data Bank (PDB) is the main database of protein structures.
# Each structure has a unique 4-character PDB ID. Below are a few examples.
protein_to_pdb = {
  "insulin": "3I40",  # Human insulin – regulates glucose uptake.
  "collagen": "1BKV",  # Human collagen – provides structural support.
  "proteasome": "1YAR",  # Archaebacterial proteasome – degrades proteins.
}

protein = "collagen"  # @param ["insulin", "collagen", "proteasome"]
pdb_structure = fetch_protein_structure(pdb_id=protein_to_pdb[protein])

pdbview = py3Dmol.view(width=400, height=400)
pdbview.addModel(pdb_structure, "pdb")
pdbview.setStyle({"cartoon": {"color": "spectrum"}})
pdbview.zoomTo()
pdbview.show()

### **Numerical Representation of a Protein**


In [ ]:
# Precursor insulin protein sequence (processed into two protein chains).
insulin_sequence = (
  "MALWMRLLPLLALLALWGPDPAAAFVNQHLCGSHLVEALYLVCGERGFFYTPKTRREAEDLQVGQVELGG"
  "GPGAGSLQPLALEGSLQKRGIVEQCCTSICSLYQLENYCN"
)
print(f"Length of the insulin protein precursor: {len(insulin_sequence)}.")

### **One-Hot Encoding of a Protein Sequence**


In [ ]:
from dlfb.utils.display import print_short_dict

# fmt: off
amino_acids = [
  "R", "H", "K", "D", "E", "S", "T", "N", "Q", "G", "P", "C", "A", "V", "I",
  "L", "M", "F", "Y", "W",
]
# fmt: on

amino_acid_to_index = {
  amino_acid: index for index, amino_acid in enumerate(amino_acids)
}

print_short_dict(amino_acid_to_index)

In [ ]:
# Methionine, alanine, leucine, tryptophan, methionine.
tiny_protein = ["M", "A", "L", "W", "M"]

tiny_protein_indices = [
  amino_acid_to_index[amino_acid] for amino_acid in tiny_protein
]

tiny_protein_indices

In [ ]:
import jax

one_hot_encoded_sequence = jax.nn.one_hot(
  x=tiny_protein_indices, num_classes=len(amino_acids)
)

print(one_hot_encoded_sequence)

In [ ]:
import seaborn as sns

fig = sns.heatmap(
  one_hot_encoded_sequence, square=True, cbar=False, cmap="inferno"
)
fig.set(xlabel="Amino Acid Index", ylabel="Protein Sequence");

Now that we’ve constructed a basic numerical representation of a protein, we’re ready
to move beyond this simplistic format and explore learned embeddings, dense vector
representations that encode much more biological meaning about each amino acid.

### **Access Learned Embeddings of Amino Acids**

This section introduces the practical workflow for utilizing state-of-the-art Protein Language Models (PLMs) to generate biological embeddings.

* **ESM2 (Evolutionary Scale Modeling)**: We will use ESM2 model to create the protein embeddings. ESM2 is a landmark protein language model, which was released by Meta in 2023. It is designed to capture the "evolutionary grammar" of protein sequences. See [ESM2](#esm2) for detailed information. ESM2 is built on the Transformer neural network architecture, which has been the industry standard for sequence modeling (both text and proteins) since 2017.

* **Hugging Face Hub**: Hugging Face Hub serves as a central repository for thousands of pretrained models across various domains. We will download and access ESM2 models from this repository.

* **Deep Learning Frameworks**:
    * **PyTorch** will be used initially to load the ESM2 model and extract embeddings, as the official weights are currently only available in this framework.
    * **JAX/Flax** will be used for all subsequent processing, analysis, and building custom classifiers on top of the extracted embeddings.

    >**Note**: Modern machine learning workflows often require "framework flexibility": extracting features in one library and performing high-performance computation in another.

#### **Logging into Hugging Face Hub**

Google Colab has a "Secrets" vault (the key icon in the left sidebar of the web UI) where we can store a Hugging Face token (`HF_TOKEN`). We get a Read (or Write) token from [Hugging Face](huggingface.co/settings/tokens) and manually add it to `HF_TOKEN` environment variable. Stored token can be then fetched using `google.colab.userdata.get()` and used to log into Hugging Face hub:

In [ ]:
import os, sys
from huggingface_hub import login

if "google.colab" in sys.modules:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')

else:
    from dotenv import load_dotenv
    load_dotenv()
    token = os.getenv("HF_TOKEN")

login(token)

Now we can access and download the model's tokenizer and pretrained weights.

Tokenizer would convert protein strings into a sequence of integer IDs that the model understands.

We will use `esm2_t33_650M_UR50D` checkpoint at this point to extract embeddings for each amino acid. This checkpoint refers to an ESM2 model with 33 layers and 650 million parameters, which was trained on UniRef50 (UniProt Referenece) database (see [here](https://www.uniprot.org/uniref) for more info).

In [ ]:
# These automatically detect the correct architecture (ESM) based on the checkpoint name.
from transformers import AutoTokenizer, EsmModel

# Define the model identifier from the Hugging Face Hub.
model_checkpoint = "facebook/esm2_t33_650M_UR50D"

# Load the tokenizer.
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

# Load the pre-trained weights into an EsmModel instance.
# By default, this loads the PyTorch version of the weights.
model = EsmModel.from_pretrained(model_checkpoint)

#### **Technical Details of Loading a Transformer Model from Hugging Face**

* **The Checkpoint Name:** The `650M` version is a "mid-sized" model. For Google Colab's free tier, this is often the upper limit of what we can fit in RAM without crashing. If we hit "Out of Memory" (OOM) errors, we can consider switching to `t12_35M` or `t6_8M`.

* **The "Auto" Logic:** Using `AutoTokenizer` is a best practice. If we decide to swap ESM2 for another model (like ProtBERT) later, we only have to change the `model_checkpoint` string; the rest of our code remains compatible.

* **Implicit Download:** The first time we run this, transformers will download several gigabytes of data to `~/.cache/huggingface/`. We should make sure that we have enough disk space if running locally.

#### **EsmTokenizer**
Now, let's see what's in the tokenizer and the base model:

In [ ]:
print(tokenizer)

The `EsmTokenizer` contains several key attributes that define how protein sequences are processed. Here is a breakdown of the most important ones:

* `name_or_path`: The identifier or local directory of the specific ESM2 model checkpoint.
* `vocab_size`: The total number of unique tokens the model understands (33 total: 20 natural amino acids, 5 non-standard/unnatural amino acids, and 8 special tokens). we can access model vocabulary using the toknizer's `get_vocab()` method.
* `model_max_length`: A theoretical limit for sequence processing, which often defaults to a placeholder for infinity.

> **Note:** The very large value often seen in `model_max_length` represents a "unlimited" placeholder. This exists because ESM2 utilizes *Rotary Positional Embeddings (RoPE)*. To learn how RoPE works, refer to [blog](https://towardsdatascience.com/understanding-positional-embeddings-in-transformers-from-absolute-to-rotary-31c082e16b26/) and if you're more interested in the topic, refer to original [research paper](https://arxiv.org/pdf/2104.09864).\
Unlike traditional Transformers that use fixed-size positional encoding tables (limiting them to 512 or 1024 tokens), RoPE allows the model to theoretically handle sequences of any length by rotating the hidden representations. Since there is no hard-coded "ceiling" in the architecture, the Hugging Face tokenizer defaults to the system's maximum possible integer.\
Despite the theoretical "infinity," we are bound by two practical limits:
> 1. **GPU VRAM:** Transformer memory consumption scales quadratically, $O(L^2)$, relative to sequence length ($L$). Processing a sequence of 10,000 amino acids will exceed the VRAM of most standard GPUs.
> 2. **Biological Accuracy:** ESM2 was pre-trained on sequences typically capped at 1024 tokens. While RoPE permits longer inputs, the model’s "biological intuition" and accuracy often degrade when it encounters sequences significantly longer than those seen during training.

> To prevent memory crashes during batch processing, it is best practice to manually override this limit during initialization:
>
>```python
>tokenizer = AutoTokenizer.from_pretrained(
>    "facebook/esm2_t33_650M_UR50D",
>    model_max_length=1024
>)
>```

#### **EsmModel**

In [ ]:
print(model)

<figure style="text-align: center; width: 750px; auto-margin :0px">
<img src="https://github.com/paymantohidifar/deep-learning-for-biology-book/blob/main/notebooks/images/esm2_model.png" alt="ESM2 Model Architecture" style="width: 100%">
<figcaption style="text-align: left top-margin:9px"><strong>ESM2 Model</strong> Architecture Overview generated by Gemini.
</figcaption>
</figure>

ESM-2 is a state-of-the-art BERT-style, encoder-only Transformer designed to learn the "probabilistic grammar" of proteins.

Let's walk through the model's components by following the flow of data through its four main stages:

#### **1. Input and Initial Embeddings**
The sequence first enters the `EsmEmbeddings` layer:
* **Vocabulary:** The biological vocabulary is mapped to only 33 possible tokens. This includes the 20 standard amino acids, rare ones like selenocysteine (U), and special technical tokens such as `<cls>` (start), `<eos>` (end), `<pad>` (padding), and `<mask>` (the training target).
* **Dense Projection:** Each token is projected into a rich, 1,280-dimensional vector space ($H$) for the 650M parameter variant.

#### **2. The ESM Encoder (Transformer Stack)**
This core "brain" consists of 33 identical layers (in the 650M model).
* **Self-Attention with RoPE:** Unlike earlier models using absolute positional encoding, ESM-2 uses Rotary Position Embedding (RoPE). This allows the model to extrapolate and handle sequences longer than those seen during its training context window.
* **Feed-Forward Network (FFN):** Each position passes through a two-layer network that expands to an intermediate dimension of 5,120 ($4 \times H$) before projecting back.
* **Zero Dropout:** To maximize effective model capacity, ESM-2 removed all dropout from hidden layers and attention mechanisms, which researchers found improved performance at scale.
* **Post-Layer Normalization:** The model applies `LayerNorm` after the attention and FFN blocks, with a final normalization step before the task heads.

#### **3. Specialized Task Heads**
Depending on the objective, the encoder output is passed to parallel branches:
* **ESM Pooler:** It extracts the `<cls>` token embedding, applies a dense layer and a $Tanh$ non-linearity to create a fixed-length summary of the entire protein for sequence-level tasks like localization prediction.
* **Contact Prediction Head:** This head performs a logistic regression over the symmetrized and corrected internal attention maps. It predicts the probability that two residues are in physical contact (C-$\alpha$ distance $< 8\text{Å}$).
* **Masked LM Head:** During pre-training, this head predicts the identity of the 15% of amino acids that were randomly masked out.

#### **4. Training and Emergence**
* **Data:** The model was trained on the UniRef50 database (Sept 2021), sampling millions of sequences across evolutionary diversity.
* **Structural Emergence:** A critical finding is that as the model scales (up to 15 billion parameters), its understanding of protein sequences (measured by perplexity) correlates perfectly (up to -1.00) with its ability to predict high-resolution atomic structures.

Now that we have a high-level understanding of the ESM tokenizer and the ESM base model, let's get into the lower-level mechanics of how a Protein Language Model (PLM) maps amino acids to a continuous "latent space".

`tokenizer.get_vocab()` returns a dictionary where keys are amino acids/special tokens (e.g., 'A', 'M', '\<cls\>') and values are their corresponding unique integer indices.

In [ ]:
# Extract the full vocabulary mapping from the tokenizer.
vocab_to_index = tokenizer.get_vocab()
print_short_dict(vocab_to_index)

The table below describes some of these special tokens and how they function within a PLM context:

| Token | Name | Function |
| --- | --- | --- |
| **`<cls>`** | **Classification** | The "Header." Placed at the very start ($Index$ $0$). Its final vector is used as a summary of the entire protein's properties. |
| **`<eos>`** | **End of Sequence** | The "Period." Marks the end of the amino acid chain so the model knows the sequence is complete. |
| **`<pad>`** | **Padding** | The "Filler." Added to shorter proteins in a batch so all sequences have the same length for GPU processing. |
| **`<unk>`** | **Unknown** | The "Placeholder." Used when the model encounters an ambiguous or non-standard amino acid (like $X$ or $Z$) not in its $20$-standard vocabulary. |

Every protein we pass to the model is automatically wrapped into: `[<cls>] + [Amino Acids] + [<eos>]`.

To start, we will use the ESM2 tokenizer to encode our tiny amino acid sequence:

In [ ]:
tokenized_tiny_protein = tokenizer("MALWM")["input_ids"]
print(tokenized_tiny_protein)

# Drop the special start <cls> and <eos> tokens.
print(tokenized_tiny_protein[1:-1])

We can extract the learned token embedding matrix (first layer of ESM2 model, see above) from the model using `model.get_input_embeddings()`. This call should give us a matrix of size $33 \times 1280$, where 33 is number of tokens and 1280 is the size of the embedding vector for each token:

In [ ]:
# model.get_input_embeddings(): Accesses the 'nn.Embedding' layer of the ESM-2 model.
# .weight: Grabs the actual parameter tensor (matrix) containing the embedding vectors.
# .detach(): Disconnects the tensor from the computational graph (stops gradient tracking).
# .numpy(): Converts the PyTorch tensor into a standard NumPy array for easier analysis.
token_embeddings = model.get_input_embeddings().weight.detach().numpy()

# (Vocabulary Size, Embedding Dimension)
print(token_embeddings.shape)
print("First 10 features of Lucine (index:3)\n", token_embeddings[3, 0:10])

#### **Visualizing the Protein Embedding Space**

Each of the 33 possible tokens in the ESM2 vocabulary exists in a 1,280-dimensional space. To interpret how the model clusters these tokens, we use **t-SNE** (t-distributed Stochastic Neighbor Embedding) to compress these dimensions into two coordinates.

If we plot these tokens in reduced two dimentions, tokens that the model considers "biochemically similar" or "contextually related" will appear closer together without explicit supervision.

In [ ]:
import pandas as pd
from sklearn.manifold import TSNE
import seaborn as sns

# Initialize t-SNE to project data into 2 components (x, y)
tsne = TSNE(n_components=2, random_state=42)

# Fit and transform the high-dimensional token embeddings
embeddings_tsne = tsne.fit_transform(token_embeddings)

# Store the 2D coordinates in a DataFrame for plotting
embeddings_tsne_df = pd.DataFrame(
    embeddings_tsne, columns=["first_dim", "second_dim"]
)

# Output shape check: (33, 2)
print(embeddings_tsne_df.shape)

# Visualize the projected embeddings
fig = sns.scatterplot(
    data=embeddings_tsne_df, x="first_dim", y="second_dim", s=40
)
fig.set_xlabel("First Dimension")
fig.set_ylabel("Second Dimension");

To sanity-check whether similar types of tokens cluster in the 2D embedding space, we can label each token using known amino acid properties and replot the t-SNE projection:

In [ ]:
# Import a utility specifically designed to prevent overlapping text labels.
# It iteratively pushes labels apart until they are all legible.
from adjustText import adjust_text

# Add the actual character tokens (e.g., 'M', 'A', '<cls>') to the DataFrame.
# This ensures each row in the 2D space is linked back to its biological identity.
embeddings_tsne_df["token"] = list(vocab_to_index.keys())

# Define biological 'ground truth' categories.
# This dictionary groups amino acids by their side-chain properties
# (e.g., charge, hydrophobicity).
token_annotation = {
  "hydrophobic": ["A", "F", "I", "L", "M", "V", "W", "Y"],
  "polar uncharged": ["N", "Q", "S", "T"],
  "negatively charged": ["D", "E"],
  "positively charged": ["H", "K", "R"],
  "special amino acid": ["B", "C", "G", "O", "P", "U", "X", "Z"],
  "special token": ["-", ".", "<cls>", "<eos>", "<mask>", "<null_1>", "<pad>", "<unk>"],
}

# Map each token to its category label.
# This nested comprehension creates a lookup (e.g., 'A' -> 'hydrophobic').
embeddings_tsne_df["label"] = embeddings_tsne_df["token"].map(
  {t: label for label, tokens in token_annotation.items() for t in tokens}
)

# View the dataframe
print(embeddings_tsne_df)

In [ ]:
# Create the scatterplot with semantic styling.
# hue: Colors points by category. style: Uses different shapes for categories.
ax = sns.scatterplot(
  data=embeddings_tsne_df,
  x="first_dim",
  y="second_dim",
  hue="label",
  style="label",
  s=50,
)
ax.set_xlabel("First Dimension")
ax.set_ylabel("Second Dimension")

# Generate a list of text objects for every point in the plot.
texts = [
  ax.text(point["first_dim"], point["second_dim"], point["token"])
  for _, point in embeddings_tsne_df.iterrows()
]

# Use adjust_text to reposition the labels.
adjust_text(
  texts,
  expand=(1.5, 1.5), # Controls the 'repulsion' force between labels.
  arrowprops=dict(arrowstyle="->", color="grey")
);

We see that how the ESM2 model internalizes biochemistry through its embedding space, moving from raw numbers to biological intuition.

The t-SNE visualization confirms that the model groups amino acids by their chemical behavior without being explicitly told to do so:

* **Hydrophobic residues** (like Phenylalanine **F**, Tyrosine **Y**, and Tryptophan **W**) form distinct clusters, typically grouping away from hydrophilic ones.
* **Functional tokens** (like `<cls>` for sequence starts and `<eos>` for ends) cluster together because they share similar "structural" roles rather than chemical ones.

This internal organization proves the model has successfully learned the "grammar" of proteins, understanding that certain amino acids are interchangeable or functionally related based on their evolutionary context.

Having verified these representations, let's dive into architectural mechanics of how the ESM2 transformer actually learns these patterns during training.

### **The ESM2 Protein Language Model**

ESM2 is pre-trained through Masked Language Modeling (MLM). During training, around 15% of the amino acids in a sequence are randomly hidden behind a `<mask/>` token. To succeed, the model must analyze the surrounding residues to predict the missing one, effectively learning structural and evolutionary rules without explicit supervision.

In practice, we can test this by masking a specific residue in a protein like *insulin*. When tokenizing this sequence, the index of the masked residue shifts by $+1$ (from 29 to 30) because the tokenizer prepends a `<cls>` token to the start.

To get a prediction, we switch from the base `EsmModel` to `EsmForMaskedLM`, which includes the necessary "prediction head" to output amino acid probabilities. For rapid prototyping in Colab, we should swap the heavy *650M* parameter model for a more agile *150M* variant, trading a bit of accuracy for significantly faster inference.

In [ ]:
insulin_sequence = (
  "MALWMRLLPLLALLALWGPDPAAAFVNQHLCGSHLVEALYLVCGERGFFYTPKTRREAEDLQVGQVELGG"
  "GPGAGSLQPLALEGSLQKRGIVEQCCTSICSLYQLENYCN"
)

masked_insulin_sequence = (
  # Let's mask the `L` amino acid in the 29th position (0-based indexing):
  #       ...LALLALWGPDPAAAFVNQH  L   CGSHLVEALYLVCGERGFF...
  "MALWMRLLPLLALLALWGPDPAAAFVNQH<mask>CGSHLVEALYLVCGERGFFYTPKTRREAEDLQVGQVELGG"
  "GPGAGSLQPLALEGSLQKRGIVEQCCTSICSLYQLENYCN"
)

# Tokenize the masked insulin sequence.
masked_inputs = tokenizer(masked_insulin_sequence)["input_ids"]

# Check that we indeed have a <mask> token in the place that we expect it. Note
# that the tokenizer adds a <cls> token to the start of the sequence, so we in
# fact expect the <mask> token at position 30 (not 29).
assert masked_inputs[30] == vocab_to_index["<mask>"]

In [ ]:
from transformers import EsmForMaskedLM

model_checkpoint = "facebook/esm2_t30_150M_UR50D"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
masked_lm_model = EsmForMaskedLM.from_pretrained(model_checkpoint)

In [ ]:
print(tokenizer)

In [ ]:
print(masked_lm_model)

#### **Architecture of `ESMForMaskedLM` Model**

<figure style='text-align:center; width:750px; auto-margin:0px'>
<img src="https://github.com/paymantohidifar/deep-learning-for-biology-book/blob/main/notebooks/images/esm2_vs_esm_ml.png"
        alt="ESMForMaskedLM Architecture"
        style="width:100%">
<figcaption style="text-align: left top-margin:9px"><strong>ESMForMaskedLM Model</strong> Architecture Overview generated by Gemini.</figcaption>
</figure>

One of the main differences between ESM2 and ESMForMaskedLM is the main "decoder" that allows the model to fill in masked positions. Briefly, the encoded output $(L, 640)$ passes through a dense layer, followed by layer normalization. The final step is a decoder $(Linear(640->33))$. This computes logits over all 33 possible tokens in the vocabulary for every position. The token with the highest logit is the model's prediction for what amino acid should go there (filling the mask).

We'll feed masked insulin protein sequence to the model and extract logits from the model's output. Then, we will compute the probabilities using $Softmax$ from `jax.nn` module and plot out the values for each token.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Pass the masked sequence through the Masked Language Model (MLM).
# **tokenizer(...): Unpacks the 'input_ids' and 'attention_mask' directly into the model.
# return_tensors="pt": Ensures the output is in PyTorch format.
model_outputs = masked_lm_model(
  **tokenizer(text=masked_insulin_sequence, return_tensors="pt")
)

# Extract the 'logits' from the model output.
# Logits are raw scores (unnormalized values) for every token in the vocabulary.
model_preds = model_outputs.logits
print("Shape of model output:", model_preds.shape)

# Target the specific <mask> token.
# [0, 30]: 0 is the batch index; 30 is the sequence position (the <mask>).
# .detach().numpy(): Removes the vector from the gradient graph and converts to a NumPy array.
mask_preds = model_preds[0, 30].detach().numpy()
print("Index of maximum logit:", np.argmax(mask_preds))

# Convert raw scores to probabilities using Softmax.
# Note: Since we're using JAX here, we ensure the inputs are compatible with JAX arrays.
mask_probs = jax.nn.softmax(mask_preds)

# Visualization setup.
letters = list(vocab_to_index.keys())  # Get the labels (A, R, N, D, etc.)
fig, ax = plt.subplots(figsize=(6, 4))

# Create a bar chart.
# Each bar represents the model's confidence that a specific amino acid is the missing one.
plt.bar(letters, mask_probs, color="grey")
plt.xticks(rotation=90)
plt.title("Model Probabilities for the Masked Amino Acid.");

As shown on the plot, the model predicts Leucine for masked position correctly.

Let’s rewrite this code as a more general form as `MaskPredictor`, with methods that mask a sequence, make a prediction, and plot the predictions:


```python
# The logic is available from `dlfb.proteins.inspect`
from dlfb.proteins.inspect import MaskPredictor

display([MaskPredictor])
```

In [ ]:
# import static type hints
from transformers import PreTrainedModel, PreTrainedTokenizer
from matplotlib.figure import Figure


class MaskPredictor:
    """Predict masked amino acids using a protein language model."""

    def __init__(self, tokenizer: PreTrainedTokenizer, model: PreTrainedModel):
        """Initialize with a tokenizer and pretrained model."""
        # Stores the Hugging Face components to the instance
        self.tokenizer = tokenizer
        self.model = model

    def plot_predictions(self, sequence: str, mask_index: int) -> Figure:
        """Plot predicted probabilities for the masked amino acid."""
        # Get the probability distribution for the masked position
        mask_probs = self.predict(sequence, mask_index)

        # Setup the Matplotlib visualization
        fig, _ = plt.subplots(figsize=(6, 4))
        plt.bar(list(self.tokenizer.get_vocab().keys()), mask_probs, color="grey")
        plt.xticks(rotation=90)

        # Use f-strings to dynamically label the true residue for comparison
        plt.title(
            "Model Probabilities for the Masked Amino Acid\n"
            f"at Index={mask_index} (True Amino Acid = {sequence[mask_index]})."
        )
        return fig

    def predict(self, sequence: str, mask_index: int) -> jax.Array:
        """Return model probabilities for masked amino acid at a position."""
        # Generate the masked string (e.g., "MA<mask/>WM")
        masked_sequence = self.mask_sequence(sequence, mask_index)

        # Tokenize and move to PyTorch ("pt")
        masked_inputs = self.tokenizer(masked_sequence, return_tensors="pt")

        # Inference: Get raw logit scores from the model
        model_outputs = self.model(**masked_inputs)

        # Extract the specific token prediction.
        # Index is 'mask_index + 1' to account for the prepended <cls> token.
        mask_preds = model_outputs.logits[0, mask_index + 1].detach().numpy()

        # Convert to probability distribution using Softmax
        mask_probs = jax.nn.softmax(mask_preds)
        return mask_probs

    @staticmethod
    def mask_sequence(sequence: str, mask_index: int) -> str:
        """Insert mask token at specified index in the input sequence."""
        # Boundary check to prevent slicing errors
        if mask_index < 0 or mask_index > len(sequence):
            raise ValueError("Mask index outside of sequence range.")

        # String slicing to replace the target residue with the <mask> tag
        return f"{sequence[0:mask_index]}<mask/>{sequence[(mask_index + 1):]}"

In [ ]:
print(f"Masked amino acid: {insulin_sequence[26]}")

MaskPredictor(tokenizer, masked_lm_model).plot_predictions(
  sequence=insulin_sequence, mask_index=26
);

As illustrated above, the model is uncertain: it assigns moderate probability to several possible amino acids, indicating that this position is harder to predict based on surrounding context.

Let's deep-dive into it and see what insights we can learn from this uncertainity:

* **Low Confidence as a Feature**: If the model spreads its probability across multiple amino acids rather than picking one clearly, it isn't necessarily a "failure". Instead, it indicates biochemical flexibility.
* **Permissive Positions**: These are regions in a protein that can tolerate multiple different amino acids without losing function. The model correctly identifies these "permissive" zones, which are often found on the protein surface or in disordered, unstructured regions.

The model’s ability to predict a distribution of likely residues proves it has mastered the "probabilistic grammar" of evolution, recognizing which substitutions are chemically compatible.

The next question is how can we leverage this understanding to represent an entire protein, and not just one amino acid at a time.

### **Strategies for Extracting an Embedding for an Entire Protein**

This section addresses a core challenge of how to transform a biological sequence of any length into a standardized "feature vector" that a machine learning model can process. Several strategies are commonly used:

#### **1. Concatenation of Amino Acid Embeddings: The "Naive" Approach**

This method creates a single massive vector by laying amino acid embeddings side-by-side. This approach poses three main problems:

* **Variable length:** For a protein of length $L$ and embedding size $D$, the result is a vector of size $L \times D$. Therefore, different proteins would have different embedding size. This fails the "Fixed-Length Requirement". Most neural networks (like a simple MLP) expect the same number of input features every time.

* **Scalability:** Long proteins produce huge embeddings. If Titin (34,000 residues) and Insulin (51 residues) are both in our dataset, we cannot easily feed them into the same architecture.

* **Limited modeling:** This approach treats amino acids independently, ignoring the contextual relationships that are central to protein function.

#### **2. Simple Averaging: The "Snapshot"**

Taking the mean of independent amino acid vectors produces a fixed-length vector (e.g., always 640 dimensions), but it won't yield a biologically accurate representation of the protein.

#### **3. Contextual Sequence Embeddings: The Gold Standard**

This is the most powerful method. Because ESM2 uses Self-Attention, its internal layers encode rich, contextualized embeddings for every amino acid in the sequence: the embedding for a Leucine at index 5 is influenced by the Tryptophan at index 50.

The process is we pass the sequence through the model and extract the final hidden layer activations to get a tensor of shape $(L^{\prime}, D)$, where $L^{\prime}$ is the number of output tokens (which may differ from the input length $L$), and D is the model’s hidden size (e.g., 640). Then, we perform Mean Pooling (averaging) across the length dimension to produce a fixed-length embedding of shape $(D,)$.

Even though we are still averaging at the final step, we aren't averaging "dumb" vectors. We are averaging vectors that already contain information about the entire protein structure. This effectively summarizes the "context" of the whole molecule into a single $D$-dimensional point.

### **Extracellular Versus Membrane Protein Embeddings**

Now that we have learned how to extract a good representation (embeddings) of a protein, we can go ahead and use it to predict whether the protein is a membrane protein or is secreted outside of cell.

To do that we will use GO (Gene Ontology) to associate each UniProt protein accession and sequence with its known cellular location. In this process, we will remove specific GO terms that are too generic for a predictive model to learn anything. For example, the following GO terms are too generic:

* *GO:0005575*: "Cellular Component" (The top-level root term)
* *GO:0110165*: "Cellular Anatomical Entity" (A very broad placeholder)

In [ ]:
import pandas as pd
from dlfb.utils.context import assets

# Load Dataset: assets() resolves the internal path to the CSV containing
# protein sequences and labels.
protein_df = pd.read_csv(assets("proteins/datasets/sequence_df_cco.csv"))
print("Original dataset size:\n", len(protein_df))

# Filter Gene Ontology (GO) Terms
protein_df = protein_df[~protein_df["term"].isin(["GO:0005575", "GO:0110165"])]
print("Filtered dataset size:\n", len(protein_df))

# 3. Calculate Dataset Statistics
num_proteins = protein_df["EntryID"].nunique()
print("Number of unique protein entries:\n", num_proteins)

In [ ]:
protein_df.head()

For each protein sequence identified by an EntryID, the "term" column provides its GO annotation for cellular localization. Let’s focus on two specific locations:

* **extracellular (GO:0005576):** Proteins secreted outside the cell, often involved in signaling, immune response, or structural roles
* **membrane (GO:0016020):** Proteins embedded in or associated with cell membranes, frequently functioning in transport, signaling, or cell–cell interaction


>**The Connection Between Cell Location and Sequence Features**: The fundamental "connection" is that protein sequences contain specific structural motifs that dictate where they locate. Since ESM2 learns the deep grammar of these sequences, it should, in theory, recognize these patterns even without being explicitly taught about "cells" or "membranes". For example, membrane proteins require hydrophobic stretches to anchor themselves into the oily lipid bilayer. On the other hand, extracellular proteins often carry signal peptides (Short leader sequence) at N-terminal of the protein.

Proteins can be expressed on multiple locations. Therefore, we first need to filter the dataset to proteins annotated with only one location:

In [ ]:
# If a protein is mapped to multiple locations, it's removed to avoid noisy labels.
num_locations = protein_df.groupby("EntryID")["term"].nunique()

# Create an Index of EntryIDs where the unique location count is exactly 1.
proteins_one_location = num_locations[num_locations == 1].index

# Filter the original DataFrame using the .isin() method to keep only 'pure' proteins.
protein_df = protein_df[protein_df["EntryID"].isin(proteins_one_location)]

print("Number of remaining proteins:\n", protein_df.shape)

As shown above, there are only 421 proteins that map into one location based on GO term values.

Now, let's find out about the length distibution of these proteins:

In [ ]:
fig = plt.figure(figsize=(8, 4))
plt.subplot(1, 2, 1)
sns.histplot(protein_df["Length"], bins=20)
plt.subplot(1, 2, 2)
sns.boxplot(protein_df["Length"], showfliers=False)

plt.tight_layout()
plt.show()

As illustrated in the plots above, around 75% of the proteins are shorter that 500 amino acids.

To manage speed and memory, we will restrict our dataset to proteins ranged from 100 to 500 amino-acid residues:

In [ ]:
# --- DEFINE SAMPLING TARGETS ---
# Map human-readable labels to specific Gene Ontology (GO) accessions.
go_function_examples = {
  "extracellular": "GO:0005576",
  "membrane": "GO:0016020",
}

# Cap sequence length for speed and memory.
min_length = 100
max_length = 500
num_samples = 20

sequences_by_function = {}

# --- DATA EXTRACTION & SAMPLING ---
for function, go_term in go_function_examples.items():
  proteins_with_function = protein_df[
    (protein_df["term"] == go_term)
    & (protein_df["Length"] >= min_length)
    & (protein_df["Length"] <= max_length)
  ]

  # Log sampling metadata for reproducibility.
  print(
    f"Found {len(proteins_with_function)} human proteins\n"
    f"with the molecular function '{function}' ({go_term}),\n"
    f"and {min_length}<=length<={max_length}.\n"
    f"Sampling {num_samples} proteins at random.\n"
  )

  # Perform random sampling with a fixed 'random_state' to ensure deterministic output.
  # Extract only the 'Sequence' column and cast it to a list for the final dictionary.
  sequences = list(
    proteins_with_function.sample(num_samples, random_state=42)["Sequence"]
  )

  # Populate the result dictionary for downstream modeling/analysis.
  sequences_by_function[function] = sequences

We’ll now use `get_mean_embeddings` to extract embeddings using a smaller ESM2 model, which produces 320-
dimensional representations and requires significantly less memory than larger
variants:

In [ ]:
from dlfb.proteins.dataset import get_mean_embeddings
from dlfb.proteins.utils import get_device

# display([get_mean_embeddings])

Here is the annotated `get_mean_embeddings` for clarity:

```python
def get_mean_embeddings(
  sequences: list[str],
  tokenizer: PreTrainedTokenizer,
  model: PreTrainedModel,
  device: torch.device | None = None,
) -> np.ndarray:
  """Compute mean embedding for each sequence using a protein LM."""
  
  # Device Management
  # Ensures the code uses a GPU (cuda/mps) if available, otherwise defaults to CPU.
  if not device:
    device = get_device()

  # Batch Tokenization & Padding
  # padding=True: Adds <pad> tokens so all sequences in the list match the length of the longest one.
  # return_tensors="pt": Returns PyTorch tensors instead of Python lists.
  model_inputs = tokenizer(sequences, padding=True, return_tensors="pt")

  # Move Data to Device
  # Dictionary comprehension to move 'input_ids' and 'attention_mask' to the GPU/CPU.
  model_inputs = {k: v.to(device) for k, v in model_inputs.items()}

  # Model Preparation
  # model.to(device): Moves the weights to the active device.
  # model.eval(): Disables layers like Dropout to ensure deterministic outputs during inference.
  model = model.to(device)
  model.eval()

  # The Forward Pass
  # torch.no_grad(): Disables the "gradient tape," saving significant memory and compute time.
  with torch.no_grad():
    outputs = model(**model_inputs)
    
    # Mean Pooling (The "Compression" Step)
    # outputs.last_hidden_state has shape (Batch, Sequence_Length, Hidden_Size).
    # .mean(dim=1): Averages across the Sequence_Length dimension.
    # Result: (Batch, Hidden_Size).
    mean_embeddings = outputs.last_hidden_state.mean(dim=1)

  # Cleanup & Format Conversion
  # Moves the result back to CPU memory and converts to a NumPy array for easier analysis.
  return mean_embeddings.detach().cpu().numpy()
  
  ```

For demonstration, we will use the smallest ESM-2 model:

In [ ]:
model_checkpoint = "facebook/esm2_t6_8M_UR50D"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = EsmModel.from_pretrained(model_checkpoint)

In [ ]:
# Compaute mean protein embeddings for each location.
protein_embeddings = {
  loc: get_mean_embeddings(sequences_by_function[loc], tokenizer, model)
  for loc in ["extracellular", "membrane"]
}

# Reformat data.
labels, embeddings = [], []
for location, embedding in protein_embeddings.items():
  labels.extend([location] * embedding.shape[0])
  embeddings.append(embedding)
  print(f"{location}: {embedding.shape}")

In [ ]:
print(embeddings)

#### **A side note on `tensor.to()` method**

In PyTorch, "moving to device" ensures that both our data and our model weights reside on the same hardware. Models and tensors default to the CPU. However, the matrix math required for ESM2 is significantly faster on a GPU (CUDA or MPS).

We cannot perform operations between a CPU tensor and a GPU model. They must match. In the function shown above, the dictionary comprehension `{k: v.to(device) for k, v in model_inputs.items()}` iterates through the tokenizer's output (like `input_ids` and `attention_mask`) and "uploads" each tensor to the specified hardware.

CPU uses system RAM (larger capacity) but is slow for parallel matrix multiplication. GPU, however, uses Video RAM (VRAM with limited capacity) and is extremely fast in parallel processing of data.

VRAM is typically limited. To process large protein datasets without crashing our GPU, we should feed dataset in baches. This ensures we only "upload" a small chunk of data to the VRAM at any given time. Moreover, if our sequences vary wildly in length (e.g., 50 to 2,000 amino acids), one long protein in a batch will force every other protein to be padded with thousands of zeros. To maximize speed, we should sort our sequences by length before batching to minimize padding overhead. Here is a modified version of `get_mean_embeddings` that addresses the padding issue:


```python
import torch
import numpy as np

def get_mean_embeddings_sorted(
    sequences: list[str],
    tokenizer: "PreTrainedTokenizer",
    model: "PreTrainedModel",
    batch_size: int = 16
) -> np.ndarray:
    """Sorts sequences by length to minimize padding, then restores order."""
    
    # Length-Based Sorting (The Optimization)
    indexed_seqs = sorted(enumerate(sequences), key=lambda x: len(x[1]))
    # zip(*...) unpacks the tuples into two separate lists for processing.
    sorted_indices, sorted_seqs = zip(*indexed_seqs)
    
    # Hardware Preparation
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # Model Preparation
    model.to(device)
    model.eval()
    
    sorted_embeddings = []

    # Batch looping
    for i in range(0, len(sorted_seqs), batch_size):
        batch_seqs = sorted_seqs[i : i + batch_size]
        
        # padding=True now only pads to the longest sequence in this specific batch.
        inputs = tokenizer(batch_seqs, padding=True, return_tensors="pt").to(device)
        
        with torch.no_grad():
            outputs = model(**inputs)
            # Extract hidden states and average across the sequence length (dim=1).
            pooled = outputs.last_hidden_state.mean(dim=1).cpu().numpy()
            sorted_embeddings.append(pooled)
            
    # Global Reconstruction
    # Combine all individual batch matrices into one large array (N, Hidden_Dim).
    all_sorted_embs = np.vstack(sorted_embeddings)
    
    # Order Restoration (The "Unsort")
    # np.argsort tells us how to rearrange the sorted_indices back to 0, 1, 2...
    # This ensures the output matches the input list provided by the user.
    original_order = np.argsort(sorted_indices)
    return all_sorted_embs[original_order]

```

> 1. **VRAM Management**: By slicing `sequences[i : i + batch_size]`, the GPU only holds the hidden states for `batch_size=16` proteins at a time rather than thousands.
> 2. **The `.cpu()` Escape**: We move the result back to CPU RAM (`.cpu().numpy()`) inside the loop. This allows the GPU to flush the massive intermediate calculations ($Batch \times Length \times Hidden-dim$) to make room for the next batch.
> 3. **`np.vstack`**: Instead of growing a massive PyTorch tensor (which is memory-heavy), we store small NumPy arrays in a list and join them at the very end.

Each set of 20 sampled proteins is now represented as a (20, 320) embedding matrix. This means that for each sequence, regardless of its original length, we obtain a fixed-size vector of 320 dimensions. These vectors correspond to the mean of the final hidden layer activations across all tokens in the sequence, and should capture some information about the overall protein structure.

To visualize how these embeddings might relate to protein localization, we project them into two dimensions using t-SNE:

In [ ]:
import numpy as np
import seaborn as sns
from sklearn.manifold import TSNE

embeddings_tsne = TSNE(n_components=2, random_state=42).fit_transform(
  np.vstack(embeddings)
)
embeddings_tsne_df = pd.DataFrame(
  {
    "first_dimension": embeddings_tsne[:, 0],
    "second_dimension": embeddings_tsne[:, 1],
    "location": np.array(labels),
  }
)

fig = sns.scatterplot(
  data=embeddings_tsne_df,
  x="first_dimension",
  y="second_dimension",
  hue="location",
  style="location",
  s=50,
  alpha=0.7,
)
plt.title("t-SNE of Protein Embeddings")
fig.set_xlabel("First Dimension")
fig.set_ylabel("Second Dimension");

While the separation isn’t perfect, there’s a clear trend: extracellular proteins tend to cluster in a different region of embedding space than membrane proteins. It’s quite striking that the model picks up on this purely from sequence. This suggests that the learned embeddings reflect biologically meaningful patterns, even without any explicit supervision for cellular location.

With this initial exploration complete, we now turn to the central machine learning task of this chapter: predicting protein function. Let’s begin by preparing the dataset.

---

## **Preparing the Data**

While many jump straight to model training, professional practice reveals that success is built on data preparation. In computational biology, the root cause of model failure is rarely the architecture, but rather the quality and structure of the input data.

Our objective is to predict protein function from sequence by assembling a dataset of sequence-function pairs. We utilize the [CAFA (Critical Assessment of Functional Annotation)](https://biofunctionprediction.org/cafa/) challenge as our primary resource, a benchmark that mirrors the famous **CASP (Critical Assessment of Structure Prediction)** competition for structure prediction.

To following workflow is designed to turn this raw biological material into a model-ready format:

1. **Sequence Standardization:** Cleaning and validating amino acid strings.
2. **Functional Mapping:** Associating sequences with hierarchical GO terms.
3. **Vectorization:** Converting text-based sequences into numerical embeddings using models like ESM2.
4. **Filtering:** Removing overly broad "root" GO terms to ensure the model learns specific biological insights.

This step-by-step approach ensures that when a model struggles, we have the intuition to identify whether the flaw lies in the learning process or the biological data itself.

### **Loading the CAFA3 Data**

There have been several rounds of CAFA, but the CAFA3 dataset is the most recent publicly available one. We first downloaded the “CAFA3 Targets” and “CAFA3 Training Data” files from the CAFA website.

> **Note:** [CAFA5](https://www.kaggle.com/c/cafa-5-protein-function-prediction) and [CAFA6](https://www.kaggle.com/competitions/cafa-6-protein-function-prediction/data?select=Train) are avialable through Kaggle competition.

Let’s start by loading the label file, which tells us the functional annotations for each protein:

In [ ]:
labels = pd.read_csv(
  assets("proteins/datasets/train_terms.tsv.zip"), sep="\t", compression="infer"
)

In [ ]:
print(labels.shape)
labels.head()

The dataframe above has three columns:
* **EntryID:** UniProt ID of the protein
* **term:** A GO accession code describing a specific protein function
* **aspect:** The GO category the function belongs to (e.g. biological process (BPO), molecular function (MFO), and cellular component (CCO))

To make CAFA dataset more human-readable, we cross-reference GO terms using the official Gene Ontology library. For more details, check [this](https://geneontology.org/docs/download-go-annotations/) out.

We can use `obonet` Python library to parse `.obo` (Open Biomedical Ontology format) graph format to programmatically map each GO ID to its biological definition. Here is the Python function to retrieve and parse the files:

```python
import obonet

def get_go_term_descriptions(store_path: str) -> pd.DataFrame:
  """Return GO term to description mapping, downloading if needed."""
  if not os.path.exists(store_path):
    url = "https://current.geneontology.org/ontology/go-basic.obo"

    # --- To get around 403 error ---
    # import requests
    # import io

    # response = requests.get(url)
    # graph = obonet.read_obo(io.StringIO(response.text))
    # -------------------------------

    graph = obonet.read_obo(url)  

    # Extract GO term IDs and names from the graph nodes.
    id_to_name = {id: data.get("name") for id, data in graph.nodes(data=True)}
    go_term_descriptions = pd.DataFrame(
      zip(id_to_name.keys(), id_to_name.values()),
      columns=["term", "description"],
    )
    go_term_descriptions.to_csv(store_path, index=False)

  else:
    go_term_descriptions = pd.read_csv(store_path)
  return go_term_descriptions
  
```

In [ ]:
from dlfb.proteins.dataset import get_go_term_descriptions

# display(["import obonet", get_go_term_descriptions])

In [ ]:
go_term_descriptions = get_go_term_descriptions(
  store_path=assets("proteins/datasets/go_term_descriptions.csv")
)

In [ ]:
print(go_term_descriptions.shape)
go_term_descriptions.head()

We then merge the human-readable term descriptions back onto the labels
dataframe:

In [ ]:
labels = labels.merge(go_term_descriptions, on="term")

In [ ]:
print(labels.shape)
labels.head()

For now in this chapter, we’ll focus specifically on molecular functions (MFO). Let's take a look at the most commonly annotated molecular functions:

In [ ]:
labels = labels[labels["aspect"] == "MFO"]

In [ ]:
print(labels.shape)
labels.head()

In [ ]:
print(labels["description"].value_counts())

As shown above, the distribution of protein function annotations is heavily skewed, dominated by generic terms like "molecular_function" and "binding." These high-frequency labels provide minimal biological insight and will be filtered out to improve model performance.

To pair these labels with their respective biological data, we must load protein sequences from a FASTA file. Using the `BioPython.SeqIO` module, we can parse these sequences into a Pandas DataFrame for easier manipulation.

In [ ]:
from Bio import SeqIO

seqs = assets("proteins/datasets/train_sequences.fasta")
records = SeqIO.parse(open(seqs), "fasta")
sequence_df = pd.DataFrame(
    data=[(record.id, str(record.seq), len(record.seq)) for record in records],
    columns = ["EntryID", "Sequence", "Length"]
)

In [ ]:
print(sequence_df.shape)
sequence_df.head()

The CAFA dataset includes proteins from many different organisms. To isolate human proteins, we’ll use the associated taxonomy file provided in the download:

In [ ]:
taxonomy_file = assets("proteins/datasets/train_taxonomy.tsv.zip")
taxonomy = pd.read_csv(taxonomy_file, sep="\t", compression="infer")

In [ ]:
print(taxonomy.shape)
taxonomy.head()

In [ ]:
sequence_df = sequence_df.merge(taxonomy, on="EntryID")
sequence_df = sequence_df[sequence_df["taxonomyID"] == 9606] # human tax ID = 9606

In [ ]:
print(sequence_df.shape)
sequence_df.head()

Now, we merge labels to sequences to have full data/metadata:

In [ ]:
sequence_df = sequence_df.merge(labels, on="EntryID")

In [ ]:
print(sequence_df.shape)
sequence_df.head()

In [ ]:
print(
  f'Dataset contains {sequence_df["EntryID"].nunique()} human proteins '
  f'with {sequence_df["term"].nunique()} molecular functions.'
)

From this table, we can already see that many proteins are associated with multiple molecular functions. To quantify this, we examine the distribution of the number of functions per protein:

In [ ]:
sequence_df.groupby("EntryID")["term"].nunique().plot.hist(
  bins=100, figsize=(5, 3), color="grey", log=True
)
plt.xlabel("Number of Molecular Function Annotations per Protein")
plt.ylabel("Frequency (log scale)")
plt.title("Distribution of Function Counts per Protein")
plt.tight_layout()

The biological reality of protein "multitasking" means that many proteins perform multiple roles, acting as enzymes while simultaneously binding to other molecules. For machine learning, this creates two specific challenges: multi-label classification and extreme class imbalance (where some functions are rare and others are ubiquitous).

As we showed in an ealier step, broad terms like "molecular function" or "protein binding" are so universal that they provide almost no predictive value. To prevent the model from "cheating" by fixating on these dominant but generic labels, we must explicitly filter them out during preprocessing, forcing the model to learn more specific and meaningful biological functions.

In [ ]:
print(sequence_df["description"].value_counts())

In [ ]:
uninteresting_functions = [
  "GO:0003674",  # "molecular function". Applies to 100% of proteins.
  "GO:0005488",  # "binding". Applies to 93% of proteins.
  "GO:0005515",  # "protein binding". Applies to 89% of proteins.
]

sequence_df = sequence_df[~sequence_df["term"].isin(uninteresting_functions)]

In [ ]:
print(sequence_df.shape)

Conversely, many molecular functions are exceptionally rare; for instance, **GO:0099609** (microtubule lateral binding) appears only a single time in the dataset. To ensure the model learns statistically significant associations rather than noise, we must provide sufficient training examples for each functional category. Consequently, we will prune the rarest labels, retaining only those that appear in at least **50** distinct proteins.

* **The "Cold Start" Problem**: Deep learning models generally fail to generalize from a single example. By enforcing a minimum threshold of **50**, we ensure the model has enough "diversity" in sequences to recognize the patterns underlying a specific function.
* **Balancing the Tail**: This filtering step significantly reduces the output dimensionality of our model, transforming a potentially unmanageable multi-label problem into a focused, learnable task.

>**Note:** Thresholds used during data processing, like how many times a label must appear to be included, are somewhat arbitrary, but they can significantly affect model performance. <u>These decisions are effectively hyperparameters and should be tuned based on the specific task, dataset size, and model capacity.</u>

In [ ]:
common_functions = (
  sequence_df["term"]
  .value_counts()[sequence_df["term"].value_counts() >= 50]
  .index
)

sequence_df = sequence_df[sequence_df["term"].isin(common_functions)]

In [ ]:
print(sequence_df.shape)

In [ ]:
sequence_df["term"].value_counts()

Next, we reshape the dataframe from long to wide format by pivoting on the term variable. This transforms the categorical annotations into a binary occurrence matrix, analogous to one-hot encoding:

In [ ]:
sequence_df = (
  sequence_df[["EntryID", "Sequence", "Length", "term"]]
  .assign(value=1)
  .pivot(
    index=["EntryID", "Sequence", "Length"], columns="term", values="value"
  )
  .fillna(0)
  .astype(int)
  .reset_index()
)

In [ ]:
print(sequence_df.shape)
sequence_df.head()

This dataset is now in a format that’s almost ready for machine learning.

Before we move on, let’s run a few final sanity checks on (1) number of unique proteins (2) number of unique sequences:

In [ ]:
print("Number of unique proteins", sequence_df["EntryID"].nunique())
print("Number of unique sequences", sequence_df["Sequence"].nunique())

The number of protein in 10,000 is in right ballpark. There are roughly 21,000 protein-coding genes in the human genome, and since we applied several filtering steps, we expect a somewhat smaller number. It’s always worth keeping rough order-of-magnitude expectations in mind—if we saw 1,000 or 1,000,000 here, we’d suspect something was off.

On the other hand, number of sequences is a little less than the number of proteins, suggesting some of the proteins have identical sequences.

Let's find out which protein entries have repeated sequences:

In [ ]:
sequence_df[sequence_df["Sequence"].duplicated(keep=False)]['EntryID']

For example, the entries P0DP23, P0DP24, and P0DP25 all share the same sequence. It is important to note that while P0DP23 (CALM1), P0DP24 (CALM2), and P0DP25 (CALM3) produce protein molecules that are biochemically identical, they are not entirely functionally the same due to differences in how and where they are expressed in the body. Therefore, these seems to legitimate biological duplicates, so we will keep them in the dataset.

In [ ]:
sequence_df[sequence_df["EntryID"].isin(["P0DP23", "P0DP24", "P0DP25"])]

Since our simple mean embedding approach can be quite memory intensive, we’ll filter the dataset to include only proteins with a maximum length of 500 amino acids. This helps avoid out-of-memory (OOM) errors during model inference and training.

Let's visualize the distribution of proteins length:

In [ ]:
sns.histplot(sequence_df[sequence_df['Length']<5000]['Length'], bins=20);

In [ ]:
sequence_df = sequence_df[sequence_df["Length"] <= 500]

In [ ]:
print(sequence_df.shape)

We almost halved the dataset size which is perfectly fine for our initial prototyping.

### **Splitting the Dataset into Subsets**

To ensure reliable evaluation, we partition our proteins by `EntryID` into three mutually exclusive subsets:

* **Training Set**: Data the model observes directly to learn biological patterns (60% of dataset).
* **Validation (dev) Set**: Used during development to tune hyperparameters and select the best model version (20% of dataset).
* **Test Set**: A held-out set used exactly once for the final evaluation to estimate real-world generalization (20% of dataset).

In [ ]:
from sklearn.model_selection import train_test_split

# 60% of the proteins will go into the training set.
train_sequence_ids, valid_test_sequence_ids = train_test_split(
  list(set(sequence_df["EntryID"])), test_size=0.40, random_state=42
)

# Split the remaining 40% evenly between validation and test sets.
valid_sequence_ids, test_sequence_ids = train_test_split(
  valid_test_sequence_ids, test_size=0.50, random_state=42
)

In [ ]:
sequence_splits = {
  "train": sequence_df[sequence_df["EntryID"].isin(train_sequence_ids)],
  "valid": sequence_df[sequence_df["EntryID"].isin(valid_sequence_ids)],
  "test": sequence_df[sequence_df["EntryID"].isin(test_sequence_ids)],
}

In [ ]:
for split, df in sequence_splits.items():
  print(f"{split} has {len(df)} entries.")

### **Converting Protein Sequences into Their Mean Embeddings**

We will now convert the protein sequences from each split into their mean embeddings. To optimize this time-intensive process, we utilize two key strategies:

* **Hardware Acceleration**: Using a GPU to parallelize the transformer forward pass.
* **Persistent Storage**: Computing the embeddings once and saving them to disk (serialization) to avoid redundant computation in future sessions.

To streamline this, we implement two helper functions for storing and loading these high-dimensional vectors.

```python
def store_sequence_embeddings(
  sequence_df: pd.DataFrame,
  store_prefix: str,
  tokenizer: PreTrainedTokenizer,
  model: PreTrainedModel,
  batch_size: int = 64,
  force: bool = False,
) -> None:
  """Extract and store mean embeddings for each protein sequence."""
  model_name = str(model.name_or_path).replace("/", "_")
  store_file = f"{store_prefix}_{model_name}.feather"

  if not os.path.exists(store_file) or force:
    device = get_device()

    # Iterate through protein dataframe in batches, extracting embeddings.
    n_batches = ceil(sequence_df.shape[0] / batch_size)
    batches: list[np.ndarray] = []
    for i in range(n_batches):
      batch_seqs = list(
        sequence_df["Sequence"][i * batch_size : (i + 1) * batch_size]
      )
      batches.extend(get_mean_embeddings(batch_seqs, tokenizer, model, device))

    # Store each of the embedding values in a separate column in the dataframe.
    embeddings = pd.DataFrame(np.vstack(batches))
    embeddings.columns = [f"ME:{int(i)+1}" for i in range(embeddings.shape[1])]
    df = pd.concat([sequence_df.reset_index(drop=True), embeddings], axis=1)
    df.to_feather(store_file)


def load_sequence_embeddings(
  store_file_prefix: str, model_checkpoint: str
) -> pd.DataFrame:
  """Load stored embedding DataFrame from disk."""
  model_name = model_checkpoint.replace("/", "_")
  store_file = f"{store_file_prefix}_{model_name}.feather"
  return pd.read_feather(store_file)

```

In [ ]:
from dlfb.proteins.dataset import load_sequence_embeddings, store_sequence_embeddings

# display([store_sequence_embeddings, load_sequence_embeddings])

Here, we will use a bigger model:

In [ ]:
model_checkpoint = "facebook/esm2_t30_150M_UR50D"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = EsmModel.from_pretrained(model_checkpoint)

In [ ]:
for split, df in sequence_splits.items():
  store_sequence_embeddings(
    sequence_df=df,
    store_prefix=assets(f"proteins/datasets/protein_dataset_{split}"),
    tokenizer=tokenizer,
    model=model,
  )

In [ ]:
train_df = load_sequence_embeddings(
  assets("proteins/datasets/protein_dataset_train"),
  model_checkpoint=model_checkpoint,
)

In [ ]:
print(train_df.shape)
train_df.head()

We noticed a series of columns labeled `ME:1` through `ME:640`. These represent the **mean-pooled hidden states** from the final layer of the ESM2 model, effectively a fixed-length numerical summary of each protein sequence. These embeddings capture biochemical and structural information learned during pretraining and will serve as the input features for our classifier.

This dataframe becomes the input to a `convert_to_tfds` function, which we’ve defined to make it easier to prepare the datasets for each split:

```python
import tensorflow as tf

def convert_to_tfds(
  df: pd.DataFrame,
  embeddings_prefix: str = "ME:",   # Prefix for Mean Embedding columns (e.g., ME:0, ME:1...)
  target_prefix: str = "GO:",       # Prefix for multi-hot encoded GO labels
  is_training: bool = False,       # Toggle for training-specific optimizations
  shuffle_buffer: int = 50,         # Determines how many items to buffer for random sampling
) -> tf.data.Dataset:
  """Convert embedding DataFrame into a TensorFlow dataset."""
  
  # Slice and Extract
  # filter(regex=...) identifies columns by their prefix.
  # to_numpy() converts the tabular data into dense numerical matrices.
  # from_tensor_slices() creates a dataset where each row in the array becomes an element.
  dataset = tf.data.Dataset.from_tensor_slices(
    {
      "embedding": df.filter(regex=f"^{embeddings_prefix}").to_numpy(),
      "target": df.filter(regex=f"^{target_prefix}").to_numpy(),
    }
  )

  # Training Pipeline Optimizations
  if is_training:
    # shuffle() ensures the model doesn't learn the order of the examples.
    # repeat() allows the dataset to be streamed indefinitely over multiple epochs.
    dataset = dataset.shuffle(shuffle_buffer).repeat()

  return dataset
  
```

>**Note:** Because the training dataset includes `.repeat()`, it yields batches
indefinitely by looping over the data. This is useful for training,
where we want to cycle through the dataset multiple times. In
contrast, the validation and test datasets are not repeated, so their
batches will eventually be exhausted, which is exactly what we want
during evaluation, where each example should be seen only once.

In [ ]:
from dlfb.proteins.dataset import convert_to_tfds

# display(["import tensorflow as tf", convert_to_tfds])

In [ ]:
train_ds = convert_to_tfds(train_df, is_training=True)

Fetching a batch of data from these datasets is straightforward. We just batch the dataset, convert it to a NumPy iterator, and retrieve a batch by calling next:

In [ ]:
batch_size = 32

batch = next(train_ds.batch(batch_size).as_numpy_iterator())
batch["embedding"].shape, batch["target"].shape

To streamline the dataset setup, we’ve wrapped the entire pipeline into a single helper function, `build_dataset`:

```python
def build_dataset(
  store_file_prefix: str,
  model_checkpoint: str
) -> dict[str, tf.data.Dataset]:
  """Build train/valid/test TensorFlow datasets from stored embeddings."""
  dataset_splits = {}

  for split in ["train", "valid", "test"]:
    dataset_splits[split] = convert_to_tfds(
      df=load_sequence_embeddings(
        store_file_prefix=f"{store_file_prefix}_{split}",
        model_checkpoint=model_checkpoint,
      ),
      is_training=(split == "train"),
    )
  return dataset_splits

```

In [ ]:
from dlfb.proteins.dataset import build_dataset

# display([build_dataset])

In [ ]:
dataset_splits = build_dataset(
  assets("proteins/datasets/protein_dataset"), model_checkpoint=model_checkpoint
)

With this, we now have our data fully preprocessed and ready to use in training a model.

## **Training the Model**

We will now train a lightweight Multi-Layer Perceptron (MLP) on top of the fixed-size protein embeddings. While the original protein sequences vary in length, our pre-computed mean embeddings provide a consistent input for the model.

Our objective is to predict associations across 303 molecular functions. This is a multi-label classification task, as a single protein often performs multiple biological roles simultaneously.

Crucially, the ESM2 model remains frozen; we are not fine-tuning its internal parameters. Instead, our MLP acts as a task-specific head that learns to interpret the static representations generated by ESM2.

Here is how our MLP logic look like:

```python
import flax.linen as nn
from flax.training import train_state


class Model(nn.Module):
  """Simple MLP for protein function prediction."""

  # Hyperparameters
  num_targets: int       # Number of GO terms to predict (e.g., 303)
  dim: int = 256         # Base dimension for hidden layers

  @nn.compact
  def __call__(self, x):
    """Apply MLP layers to input features."""
    
    # Sequential Architecture
    x = nn.Sequential(
      [
        # Layer 1: Expansion. Projects embedding (e.g., 256) to a wider space (512).
        nn.Dense(self.dim * 2),
        jax.nn.gelu,          # Smooth activation (Gaussian Error Linear Unit)
        
        # Layer 2: Contraction.
        nn.Dense(self.dim),
        jax.nn.gelu,
        
        # Layer 3: Output Projection.
        # Projects to the number of functional labels.
        nn.Dense(self.num_targets),
      ]
    )(x)
    return x

  def create_train_state(self, rng: jax.Array, dummy_input, tx) -> TrainState:
    """Initialize model parameters and return a training state."""
    
    # Parameter Initialization
    # self.init runs the forward pass with dummy_input to determine weight shapes.
    variables = self.init(rng, dummy_input)
    
    # Training State Creation
    # Encapsulates parameters, the forward function (apply_fn), and the optimizer (tx).
    return TrainState.create(
      apply_fn=self.apply,
      params=variables["params"],
      tx=tx
    )

```

The above MLP model architecture and training strategy follow these principles:

* **Sequential Structure**: It utilizes `nn.Sequential` to stack layers, maintaining a clean and readable definition.
* **Activation Function**: The model employs GELU (Gaussian Error Linear Unit), a smooth, nonlinear alternative to ReLU.
* **Output Layer**: The final `nn.Dense` layer projects to `num_targets`, returning raw logits rather than probabilities.
* **Probabilistic Conversion**: Predictions are converted to probabilities using a sigmoid activation within the loss function.
* **Frozen Embeddings**: The model is trained on top of static ESM2 representations, meaning the transformer weights are not updated.
* **Efficiency**: This frozen approach reduces memory consumption and enhances training efficiency and interpretability.

In [ ]:
from dlfb.proteins.model import Model

# display(
#   ["import flax.linen as nn\nfrom flax.training import train_state\n", Model]
# )

To streamline the setup, we utilize a `create_train_state` convenience function to encapsulate model initialization, parameter registration, and optimizer configuration into a single `TrainState` object. This pattern allows us to perform shape inference using dummy data and prepare the model for training in one cohesive step.

We then instantiate the model, setting the output layer size to match the number of `GO term` labels present in our processed training data.

In [ ]:
targets = list(train_df.columns[train_df.columns.str.contains("GO:")])

mlp = Model(num_targets=len(targets))

In [ ]:
mlp

### **Defining the Training Loop**

With the model and dataset ready, we can now define a function to perform a single training step. This step includes:

* A forward pass through the model
* Computing the loss
* Calculating gradients
* Updating the model parameters using those gradients

Here’s how we implement it:

```python
@jax.jit
def train_step(state, batch):
  """Run a single training step and update model parameters."""

  # Define the Differentiable Objective
  def calculate_loss(params):
    """Compute sigmoid cross-entropy loss from logits."""
    # Pass embeddings through the model (state.apply_fn) to get raw logits
    logits = state.apply_fn({"params": params}, x=batch["embedding"])
    
    # Use Sigmoid Binary Cross Entropy for multi-label classification.
    loss = optax.sigmoid_binary_cross_entropy(logits, batch["target"]).mean()
    return loss

  # Gradient Calculation
  # value_and_grad returns both the loss (value) and the derivatives (grad).
  grad_fn = jax.value_and_grad(calculate_loss, has_aux=False)
  loss, grads = grad_fn(state.params)

  # Parameter Update
  # apply_gradients handles the optimizer logic (e.g., Adam) to update weights.
  state = state.apply_gradients(grads=grads)
  
  return state, loss

```

In [ ]:
from dlfb.proteins.train import train_step

# display([train_step])

In this setup:
* We use a sigmoid activation and binary cross-entropy loss, appropriate for multilabel classification. The logits go through a sigmoid activation, not softmax because we want independent yes/no predictions for each possible protein function. Remember that each protein could have many functions at once.
* `@jax.jit` compiles the training step for better performance.

Next, let’s implement some metrics to evaluate how well the model is doing beyond the loss alone, using tools from sklearn:

```python
import numpy as np
from sklearn import metrics

def compute_metrics(
    targets: np.ndarray, probs: np.ndarray, thresh=0.5
) -> dict[str, float]:
    """Compute accuracy, recall, precision, auPRC, and auROC."""
    
    # Edge-Case Guard: Prevent division-by-zero crashes if there are no positive targets
    # (e.g., auROC/Recall are mathematically undefined if the true positive count is 0)
    if np.sum(targets) == 0:
        return {
            m: 0.0 for m in ["accuracy", "recall", "precision", "auprc", "auroc"]
        }
        
    return {
        # Threshold-dependent metric: Evaluates hard discrete classifications (True vs False)
        "accuracy": float(metrics.accuracy_score(targets, probs >= thresh)),
        
        # Recall (Sensitivity): Proportion of actual positives correctly identified
        "recall": metrics.recall_score(targets, probs >= thresh).item(),
        
        # Precision (PPV): Proportion of predicted positives that are truly positive
        # zero_division=0.0 prevents a crash if the model predicts 0 positive instances total
        "precision": metrics.precision_score(
            targets,
            probs >= thresh,
            zero_division=0.0,
        ).item(),
        
        # Area Under the Precision-Recall Curve (auPRC): Threshold-agnostic metric
        # that evaluates prediction confidence ranking (highly robust for imbalanced data)
        "auprc": metrics.average_precision_score(targets, probs).item(),
        
        # Area Under the Receiver Operating Characteristic (auROC): Probability that a
        # randomly chosen positive sample ranks higher than a randomly chosen negative sample
        "auroc": metrics.roc_auc_score(targets, probs).item(),
    }

```

We’ll track the following evaluation metrics for each function label:

* **Accuracy**: The fraction of correct predictions across all labels. In multilabel classification with imbalanced data (like this), accuracy can be misleading—most labels are zero, so a model that always predicts “no function” would appear accurate. Still,
it’s an intuitive metric and we’ll include it for now.

* **Recall**: The proportion of actual function labels the model correctly predicted (i.e., true positives/all actual positives). High recall means the model doesn’t miss many true functions.

* **Precision**: The proportion of predicted function labels that are correct (i.e., true positives/all predicted positives). High precision means the model avoids false alarms.

* **Area under the precision-recall curve (auPRC)**: Summarizes the tradeoff between precision and recall at different thresholds. Particularly useful in highly imbalanced settings like this one.

* **Area under the receiver operating characteristic curve (auROC)**: Measures the model’s ability to distinguish positive from negative examples across all thresholds. While it’s a standard metric of discrimination ability, it can sometimes be misleading in highly imbalanced datasets, as it gives equal weight to both classes.

In a multilabel setting, we calculate these metrics for each protein function (i.e., per target/label), then average them to get a holistic view of model performance.

In [ ]:
from dlfb.proteins.train import compute_metrics

# display(["import sklearn", compute_metrics])

We apply these metrics calculations during the evaluation step `eval_step`. The evaluation computes metrics per protein in the batch. For each protein, we:
* Apply sigmoid to its 303 logits to get function probabilities
* Threshold those probabilities (e.g., at 0.5) to get binary predictions.
* Compare these to the true function labels to compute metrics like accuracy, precision, recall, auPRC, and auROC.

We repeat this for every protein in the batch and then average the resulting metrics across proteins. This tells us how well the model predicts sets of functions per protein. It does not report performance per GO term. If we wanted per-function metrics (e.g., how well the model predicts GO:0003677), we’d need to compute metrics column-wise instead.


```python
def eval_step(state, batch) -> dict[str, float]:
    """Run evaluation step and return mean metrics over targets."""
    
    # Forward Pass: Compute unnormalized log-probabilities (logits) using the model's apply function
    logits = state.apply_fn({"params": state.params}, x=batch["embedding"])
    
    # Compute the multi-label binary cross-entropy loss and average it across the batch
    loss = optax.sigmoid_binary_cross_entropy(logits, batch["target"]).mean()
    
    # Compute metrics column-by-column (per target) across the entire batch
    target_metrics = calculate_per_target_metrics(logits, batch["target"])
    
    # Construct the final dictionary: Extract scalar loss and calculate the mean
    # of each evaluation metric across all evaluated targets using a Pandas DataFrame
    metrics_summary = {
        "loss": loss.item(),
        **pd.DataFrame(target_metrics).mean(axis=0).to_dict(),
    }
    return metrics_summary


def calculate_per_target_metrics(logits, targets) -> list[dict[str, float]]:
    """Compute metrics for each discrete target class across a multi-label batch."""
    
    probs = jax.nn.sigmoid(logits)    
    target_metrics = []
    
    # Loop over every protein (example in batch) independently
    for target, prob in zip(targets, probs):
        # compute_metrics evaluates accuracy/recall/precision/PRC/ROC for a single protein
        target_metrics.append(compute_metrics(target, prob))
        
    return target_metrics

```

In [ ]:
from dlfb.proteins.train import calculate_per_target_metrics, eval_step

# display([eval_step, calculate_per_target_metrics])

In the next chunk of code, everything comes together into a `train` function, and variations of this basic setup will be repeated in every chapter. We have the training loop where we first initialize our model training state and then loop over the dataset in batches to train the model and evaluate it every so often:

```python
def train(
    state: TrainState,
    dataset_splits: dict[str, tf.data.Dataset],
    batch_size: int,
    num_steps: int = 300,
    eval_every: int = 30,
    ):
    """Train model using batched TF datasets and track performance metrics."""
    
    # Create containers to handle calculated during training and evaluation.
    train_metrics, valid_metrics = [], []
    
    # Create batched dataset to pluck batches from for each step.
    train_batches = (
        dataset_splits["train"]
        .batch(batch_size, drop_remainder=True)
        .as_numpy_iterator()
    )

    steps = tqdm(range(num_steps)) # Steps with progress bar.
    for step in steps:
        steps.set_description(f"Step {step + 1}")
        
        # Get batch of training data, convert into a JAX array, and train.
        state, loss = train_step(state, next(train_batches))
        train_metrics.append({"step": step, "loss": loss.item()})
        
        if step % eval_every == 0:
            # For all the evaluation batches, calculate metrics.
            eval_metrics = []
            for eval_batch in (
                dataset_splits["valid"].batch(batch_size=batch_size).as_numpy_iterator()
                ):
                eval_metrics.append(eval_step(state, eval_batch))
            valid_metrics.append(
                {"step": step, **pd.DataFrame(eval_metrics).mean(axis=0).to_dict()}
        )
    return state, {"train": train_metrics, "valid": valid_metrics}

```

In [ ]:
from dlfb.proteins.train import train

# display([train])

A few notes on this training loop:

* **Efficient Data Streaming**: Training data is streamed via `.as_numpy_iterator()`, with `.repeat()` ensuring an infinite loop for continuous sampling.
* **Periodic Evaluation**: The model is evaluated on the entire validation set every `eval_every` steps using metrics like auPRC and auROC.
* **Stable Metrics**: Validation results are averaged across all batches to provide a consistent estimate of model performance.
* **The `@restorable` Decorator**: To save time, this utility checks for an existing model at a specified path. If found, it skips retraining and restores the `TrainState` and saved metrics, making iterative debugging much faster.

In [ ]:
import optax

from dlfb.utils.restore import restorable

# Initiate training state with dummy data from a single batch.
rng = jax.random.PRNGKey(42)
rng, rng_init = jax.random.split(key=rng, num=2)

state, metrics = restorable(train)(
  state=mlp.create_train_state(
    rng=rng_init, dummy_input=batch["embedding"], tx=optax.adam(0.001)
  ),
  dataset_splits=dataset_splits,
  batch_size=32,
  num_steps=300,
  eval_every=30,
  store_path=assets("proteins/models/mlp"),
)

Having trained the model with the previous train call, we can now evaluate its training dynamics and performance on the validation set:

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

from dlfb.utils.metric_plots import DEFAULT_SPLIT_COLORS

fig, ax = plt.subplots(nrows=1, ncols=2, figsize=(9, 4))

# Plot training loss curve.
learning_data = pd.concat(
  pd.DataFrame(metrics[split]).melt("step").assign(split=split)
  for split in ["train", "valid"]
)

sns.lineplot(
  ax=ax[0],
  x="step",
  y="value",
  hue="split",
  data=learning_data[learning_data["variable"] == "loss"],
  palette=DEFAULT_SPLIT_COLORS,
)
ax[0].set_title("Loss over training steps.")

# Plot validation metrics curves.
sns.lineplot(
  ax=ax[1],
  x="step",
  y="value",
  hue="variable",
  style="variable",
  data=learning_data[learning_data["variable"] != "loss"],
  palette="Set2",
)
plt.legend(loc="center left", bbox_to_anchor=(1, 0.5))
ax[1].set_title("Validation metrics over training steps.");

In the left panel, we observe that both training and validation loss drop sharply within the first ~30 steps and then stabilize. This is a typical learning curve, indicating rapid convergence without substantial instability (e.g., no major spikes or
divergence). It suggests that the model—a shallow MLP operating on top of frozen pretrained embeddings—quickly captures the low-hanging signal in the data.

In the right panel, we track several evaluation metrics over time:
* Accuracy and auROC start high and remain flat, but these can be misleading in imbalanced, multilabel settings like this one. Since most function labels are negative (i.e., a protein lacks the majority of all possible functions), a model that mostly predicts zeros can still achieve a high score on these metrics. For that reason, we don’t put much weight on these metrics in this context.
* auPRC steadily improves and does not fully plateau, suggesting the model continues to learn subtle distinctions and could potentially benefit from further training (i.e., by increasing num_steps).
* Precision improves more quickly than recall, indicating the model becomes increasingly confident in its predictions but still fails to capture some true positives.

Together, these trends indicate that while most of the learning happens early on, there may still be headroom—particularly in recall and auPRC—if training were extended further or if a more powerful architecture were used.

> **Logging Tools**: To avoid the tedium of manual plotting, we could use built-in utilities like `MetricsLogger` or professional dashboards such as [TensorBoard](https://www.tensorflow.org/tensorboard), [Weights & Biases (W&B)](https://wandb.ai/site/), or [MLflow](https://mlflow.org/).

### **Examining the Model Predictions**

Now that we have a trained model, let's examine its strengths and weaknesses. To do so, we will generate the predicts for the entire validation set and store them in a dataframe for easier inspection:


In [ ]:
valid_df = load_sequence_embeddings(
  store_file_prefix=f"{assets('proteins/datasets/protein_dataset')}_valid",
  model_checkpoint=model_checkpoint,
)

# Use batch size of 1 to avoid dropping the remainder.
valid_probs = []
for valid_batch in dataset_splits["valid"].batch(1).as_numpy_iterator():
  logits = state.apply_fn({"params": state.params}, x=valid_batch["embedding"])
  valid_probs.extend(jax.nn.sigmoid(logits))

valid_true_df = valid_df[["EntryID"] + targets].set_index("EntryID")
valid_prob_df = pd.DataFrame(
  np.stack(valid_probs), columns=targets, index=valid_true_df.index
)

In [ ]:
valid_prob_df.shape

To get a high-level sense of how the model is performing, we can visualize the full prediction matrix as a heatmap next to the heatmap of true labels across all protein Entries:

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=2, figsize=(11, 4))

sns.heatmap(
  ax=ax[0],
  data=valid_true_df,
  yticklabels=False,
  xticklabels=False,
  cmap="flare",
)
ax[0].set_title("True functional annotations by protein.")
ax[0].set_xlabel("Functional category")

sns.heatmap(
  ax=ax[1],
  data=valid_prob_df,
  yticklabels=False,
  xticklabels=False,
  cmap="flare",
)
ax[1].set_title("Predicted functional annotations by protein.")
ax[1].set_xlabel("Functional category");

The above plots help build intuition about overall model behavior:

* Some protein functions appear frequently in the dataset (visible as vertical stripes), and the model tends to predict these relatively well.
* Rare functions are harder to capture—the model often misses them entirely, leading to sparse or empty columns in the predicted eatmap.
* A few functions are over-predicted, visible as faint vertical lines across many proteins, suggesting the model is overly confident for those categories.
* Many cells in the predicted matrix show intermediate color tones, which reflect more uncertain probabilities (not a confident near-0 or near-1).

Now that we have some qualititative intuition about our predictions, let's focus on each protein function individually:

In [ ]:
metrics_by_function = {}
for function in targets:
  metrics_by_function[function] = compute_metrics(
    valid_true_df[function].values, valid_prob_df[function].values
  )

overview_valid = (
  pd.DataFrame(metrics_by_function)
  .T.merge(go_term_descriptions, left_index=True, right_on="term")
  .set_index("term")
  .sort_values("auprc", ascending=False)
)
print(overview_valid)

The model’s performance varies significantly across different protein functions. For instance, it successfully identifies **G protein–coupled receptor activity** (GO:0004930) but struggles with **cytoskeletal motor activity** (GO:0003774). These results must be interpreted carefully because rare functions have fewer validation examples, and limited training data naturally restricts the model's ability to learn.

The model performance evaluation for each protein function should be considered by the following items:

* **Data Representation:** There is a strong correlation between how often a function appears in the training set and its predictive accuracy (auPRC). The model simply has more opportunities to learn the "grammar" of common functions than rare ones.
* **Structural Signals:** Functions that perform well, like GPCR or kinase activity, often involve well-conserved structural motifs (e.g., transmembrane helices). These provide clear, predictable signals in the amino acid sequence that the ESM-2 embeddings can easily capture.
* **Statistical Reliability:** Metrics for underrepresented functions can be misleading; a high score might be based on only a handful of examples, while a low score might reflect data scarcity rather than the biological complexity of the task.
* **Class Imbalance:** Because most proteins lack the majority of possible functions, the data is highly skewed. A model might achieve high accuracy just by predicting "no function," making ranking-based metrics like auPRC much more important for seeing real progress.

#### **Thresholded and Continuous Evaluation Metrics**

Our evaluation metrics fall into two categories: thresholded and continuous:
* Precision and recall are computed from binary predictions—i.e., after applying a fixed threshold (typically >0.5) to the model’s output probabilities.
* auPRC and auROC are threshold independent. They assess
how well the model ranks positive examples above negatives across all possible thresholds.

Although a bit counterintuitive, it’s entirely possible for precision and recall to be 0 while auPRC and auROC remain high (as we saw in above metrics table). This happens when the model assigns higher probabilities to the correct labels, but those probabilities never exceed the decision threshold (e.g. 0.5 in our case). In such cases, thresholded metrics show failure, while ranking-based metrics still reflect meaningful signal.

If we wanted to address this issue with the current thresholded metrics, we could lower the decision threshold—for example, to 0.2 or 0.3—to encourage more positive predictions. The threshold can be tuned automatically using metrics like the F1 score
(the harmonic mean of precision and recall).

The following cell computes the optimal threshold for one of the fucntions with high auROC/auPRC but zero recall/precision:

In [ ]:
from sklearn.metrics import precision_recall_curve
import numpy as np

# Choose a protein function
function = "GO:0004930"

# Compute precisions, recalls, and thresholds
precisions, recalls, thresholds = precision_recall_curve(valid_true_df[function].values, valid_prob_df[function].values)

# Calculate F1-score for each threshold
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-10)
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]

print(f"Optimal Threshold: {best_threshold:.4f}")
print(f"Max F1-Score: {f1_scores[best_idx]:.4f}")

Let’s take a closer look at whether there’s a relationship between how often a protein function appears in the training data and how well the model learns to predict it in the validation set:

In [ ]:
# Compute number of occurences of each function in the training set.
overview_valid = overview_valid.merge(
  pd.DataFrame(train_df[targets].sum(), columns=["train_n"]),
  left_index=True,
  right_index=True,
)
print(overview_valid)

In [ ]:
fig = sns.scatterplot(
  x="train_n", y="auprc", data=overview_valid, alpha=0.5, s=50, color="grey"
)
fig.set_xlabel("# Train instances")
fig.set_ylabel("Validation auPRC");

The plot shows a direct correlation between how often a protein function appears in the training set and how accurately the model predicts it (auPRC). This reflects a common machine learning reality: models excel on well-represented classes but struggle with "rare" functions due to class imbalance—a lack of sufficient examples rather than inherent biological difficulty. To determine if a score like 0.8 is actually meaningful, it must be compared against a baseline to ensure the model is learning real patterns rather than just guessing based on frequency.

### **Evaluating Model Usefulness**

To ground our evaluation, we’ll compare our model against two simple baselines:

* **Coin flip**: For each protein function, randomly predict 0 or 1 with equal probability. This gives us a baseline for total ignorance.

* **Proportional guessing**: Predict 1 for each function with probability equal to its frequency in the training set. This reflects prior class distribution knowledge, but without any learning.

These baselines help contextualize the model’s performance. If our trained model doesn’t outperform these simple heuristics, it’s a sign that it may not have learned meaningful structure from the data.

Here are implementations for the baselines:

In [ ]:
def make_coin_flip_predictions(
  valid_true_df: pd.DataFrame, targets: list[str]
) -> pd.DataFrame:
  """Make random coin flip predictions for each protein function."""
  predictions = np.random.choice([0.0, 1.0], size=valid_true_df.shape)
  return pd.DataFrame(predictions, columns=targets, index=valid_true_df.index)


def make_proportional_predictions(
  valid_true_df: pd.DataFrame, train_df: pd.DataFrame, targets: list[str]
) -> pd.DataFrame:
  """Make random protein function predictions proportional to frequency."""
  percent_1_train = dict(train_df[targets].mean())
  proportional_preds = []
  for target_column in targets:
    prob_1 = percent_1_train[target_column]
    prob_0 = 1 - prob_1
    proportional_preds.append(
      np.random.choice([0.0, 1.0], size=len(valid_true_df), p=[prob_0, prob_1])
    )
  return pd.DataFrame(
    np.stack(proportional_preds).T, columns=targets, index=valid_true_df.index
  )

These baselines should give us simple but informative reference points. Let’s now apply these prediction methods, alongside our trained model:

In [ ]:
prediction_methods = {
  "coin_flip_baseline": make_coin_flip_predictions(valid_true_df, targets),
  "proportional_guess_baseline": make_proportional_predictions(
    valid_true_df, train_df, targets
  ),
  "model": valid_prob_df,
}

Now let’s evaluate the baselines in exactly the same way as our model—by computing per-protein metrics and averaging them:

In [ ]:
metrics_by_method = {}
for method, preds_df in prediction_methods.items():
  metrics_by_method[method] = pd.DataFrame(
    [
      compute_metrics(valid_true_df.iloc[i], preds_df.iloc[i])
      for i in range(len(valid_true_df))
    ]
  ).mean()

print(pd.DataFrame(metrics_by_method))

The model significantly outperforms both random and frequency-based baselines, particularly in precision, auPRC, and auROC. While frequency-based guessing yields a deceptively high accuracy due to class imbalance, the trained model leverages actual sequence features to make more informed predictions.

Currently, the model is "conservative but accurate"—it shows high precision but modest recall, meaning it effectively avoids "false alarms" but still misses many true positives. This behavior can be tuned for specific applications by lowering the decision threshold (e.g., from 0.5 to 0.2) to improve recall and capture more functional annotations.

Next, we’ll break down the model’s strengths and weaknesses by individual protein function and compare performance against both baselines. This allows us to see which specific functions the model predicts well—and where it struggles:

In [ ]:
auprc_by_function = {}

for method, preds_df in prediction_methods.items():
  metrics_by_function = {}

  for function in targets:
    metrics_by_function[function] = compute_metrics(
      valid_true_df[function], preds_df[function]
    )

  auprc_by_function[method] = (
    pd.DataFrame(metrics_by_function)
    .T.merge(go_term_descriptions, left_index=True, right_on="term")
    .set_index("term")
    .sort_values("auprc", ascending=False)
  )["auprc"].to_dict()

 Next, we visualize the function-level auPRC scores as a bar plot to highlight which functional categories the model handles best:

In [ ]:
best_performing = (
  pd.DataFrame(auprc_by_function)
  .merge(go_term_descriptions, left_index=True, right_on="term")
  .set_index("term")
  .sort_values("model", ascending=False)
  .head(20)
  .melt("description")
)

fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(
  x="description",
  y="value",
  hue="variable",
  data=best_performing,
)
ax.set_title("The model's 20 best performing protein functions")
ax.set_ylabel("Validation auPRC")
lt.xticks(rotation=90);


The model excels at predicting membrane and signaling functions, like GPCR and kinase activity, because they often rely on highly conserved structural motifs such as transmembrane helices or catalytic domains. These well-defined biochemical markers provide a much stronger "sequence-level signal" than more context-heavy roles. Overall, these findings prove the model can detect genuine biological patterns and substantially outperform basic heuristics.

### **Conducting a Final Check on the Test Set**

We must not touch the test set until we've fully finalized our model, including all hyperparameters, architectures, and training choices. Evaluating on the test set repeatedly can lead to overly optimistic results and undermine the validity of our findings.

We’ll make predictions on the test set of proteins in the same way we did for the validation set:

In [ ]:
eval_metrics = []

for split in ["valid", "test"]:
  split_metrics = []

  for eval_batch in dataset_splits[split].batch(32).as_numpy_iterator():
    split_metrics.append(eval_step(state, eval_batch))

  eval_metrics.append(
    {"split": split, **pd.DataFrame(split_metrics).mean(axis=0).to_dict()}
  )
print(pd.DataFrame(eval_metrics))

The test set performance closely aligns with the validation results, indicating strong generalization. Because we performed minimal tuning, we avoided the common issue of test-score degradation caused by overfitting to the validation set during development. These final, held-out metrics provide the most reliable estimate of real-world performance and are the standard for external reporting.

## **Improvements and Extensions**

This prototype demonstrates that protein function is predictable using a lightweight classifier paired with pretrained sequence embeddings. While technical and analytical upgrades are possible, you should first revisit the project's strategic foundation: define your users' specific needs, determine when the model is "good enough," and decide if specific functional classes or model interpretability are more critical than raw performance.

### **Biological and Analytical Exploration**

Even with a fixed model, we can learn a lot more by probing its behavior and comparing it to biological expectations:

* **Threshold tuning**: Our results showed that the model has high auPRC but low recall at a default probability threshold of 0.5. You could optimize this threshold (e.g., per protein
function or globally) using a metric like F1 score to find a better trade-off between precision and recall.
* **Species generalization**: The current dataset is human only, but this might be unnecessarily limited. Try including protein-function pairs from other species to see if performance
improves.
* **Function-specific performance drivers**: Why does the model do well on some functions (e.g., GPCR activity) but poorly on others (e.g., growth factor activity)? You could investigate whether function prevalence, sequence length, or other properties correlate with performance.
* **Examine protein multifunctionality**: Does the model struggle more with proteins that have many functions? Group proteins by number of annotated functions and plot performance (e.g., auPRC)
to see if there’s a trend.
* **False positives that might be real**: Find proteins where the model confidently predicts a function that isn’t labeled. Could the model be correct and the annotation missing? How might you follow this up?

### **Machine Learning Improvements**
From a machine learning perspective, here are a few directions you could explore:

* **Tune the MLP**: Our model is a small MLP on top of frozen embeddings. Try adding more layers, dropout, or batch normalization to increase capacity while controlling overfitting.
* **Alternative input encodings**: We used the mean-pooled embedding, which loses sequence order information. Try attention pooling or a small 1D CNN or transformer on top of the token-
level embeddings.
* **Feature engineering**: You could augment the input to include protein length, species (if you extend beyond human), or even simple statistics like embedding norms. These additional features might help the model distinguish protein types more effectively.
* **Train a per-function head**: Instead of predicting all functions jointly, try training separate models (or heads)
for each function. This can help when tasks are highly imbalanced or unrelated. Alternatively, you could cluster GO functions into a few categories and train one model per cluster.
* **Predict function hierarchically**: Rather than treating each function independently, you could use the GO hierarchy to add structure to predictions—for example, predicting broad function
categories first and then refining to more specific ones.
* **Try alternative base models**: You could plug in other protein language models from Hugging Face or explore combining embeddings from multiple models by concatenating them.
* **Unfreeze the language model**: The ESM2 embeddings are pretrained on a generic task. Fine-tuning the language model directly for protein function classification may boost performance,
though it requires more compute and a more involved training setup.

## **Summary**

This chapter marked our initial foray into biological deep learning by leveraging the pretrained ESM-2 model to extract human protein representations. After training a classifier for functional prediction, we addressed core biological modeling challenges, including severe class imbalance and the nuances of evaluating metrics like auPRC.

In Chapter 3, we transition from proteins to genomic data, building convolutional neural networks (CNNs) from scratch to detect regulatory motifs and functional elements in DNA.